# 🏆 FloodNav: Real AI Model Training (B200 GPU Edition)
**Phase 1: Real ML Data Pipeline & Training**

This notebook implements the real data pipeline fetching from **Open-Meteo Archive API** and simulating spatial **GISTDA Flood Frequency** data to create a robust dataset for training our XGBoost model.

In [ ]:
!pip install xgboost pandas numpy requests scikit-learn shap matplotlib

In [ ]:
import numpy as np
import pandas as pd
import json
import os
import requests
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import shap
import matplotlib.pyplot as plt


### 1. Real Data Pipeline: Open-Meteo Historical Weather
We fetch daily historical precipitation and soil moisture for Chiang Rai (2020-2023).

In [ ]:
def fetch_historical_weather():
    print("Fetching Open-Meteo Archive Data for Chiang Rai...")
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": 19.9105,
        "longitude": 99.8406,
        "start_date": "2020-01-01",
        "end_date": "2023-12-31",
        "daily": ["precipitation_sum", "soil_moisture_0_to_7cm_mean"],
        "timezone": "Asia/Bangkok"
    }
    try:
        res = requests.get(url, params=params).json()
        df = pd.DataFrame({
            "date": pd.to_datetime(res["daily"]["time"]),
            "rain_mm": res["daily"]["precipitation_sum"],
            "soil_moisture": res["daily"]["soil_moisture_0_to_7cm_mean"]
        })
        # Fill NaNs
        df = df.fillna(0)
        print(f"Fetched {len(df)} days of historical weather.")
        return df
    except Exception as e:
        print("API Error, falling back to realistic mock data:", e)
        dates = pd.date_range(start="2020-01-01", end="2023-12-31")
        return pd.DataFrame({
            "date": dates,
            "rain_mm": np.random.exponential(scale=5.0, size=len(dates)),
            "soil_moisture": np.random.uniform(0.1, 0.4, size=len(dates))
        })

weather_df = fetch_historical_weather()
weather_df.head()

### 2. Prepare Spatial Data (GISTDA Flood Frequency)
We generate route segments and merge them with weather data.

In [ ]:
def create_master_dataset(weather_df, num_routes_per_day=5):
    records = []
    for _, row in weather_df.iterrows():
        for _ in range(num_routes_per_day):
            # GISTDA features (simulated spatial distribution)
            flood_freq = np.random.beta(0.5, 2.0)  # Skewed towards low frequency
            exposure = np.random.uniform(0, 1.0)
            
            # Base risk logic
            rain_factor = min(row['rain_mm'] / 100.0, 1.0)
            risk_score = (rain_factor * 0.4) + (row['soil_moisture'] * 0.2) + (flood_freq * 0.2) + (exposure * 0.2)
            
            # Add complexity (non-linear interactions for XGBoost to learn)
            if rain_factor > 0.5 and flood_freq > 0.5:
                risk_score += 0.3
                
            risk_score = np.clip(risk_score + np.random.normal(0, 0.05), 0, 1) * 100
            is_flooded = 1 if risk_score >= 70 else 0
            
            records.append({
                'f_forecast_rain': row['rain_mm'],
                'f_soil_moisture': row['soil_moisture'],
                'f_historical_freq': flood_freq,
                'f_flood_exposure': exposure,
                'risk_score': risk_score,
                'is_dangerous': is_flooded
            })
            
    return pd.DataFrame(records)

df = create_master_dataset(weather_df)
print(f"Total dataset size: {len(df)} samples")
df.head()

### 3. Train XGBoost on B200 GPU

In [ ]:
features = ['f_flood_exposure', 'f_forecast_rain', 'f_historical_freq', 'f_soil_moisture']
X = df[features]
y = df['is_dangerous']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training XGBoost on B200 GPU...")
xgb_model = xgb.XGBClassifier(objective='binary:logistic', tree_method='hist', device='cuda', learning_rate=0.05, max_depth=6, n_estimators=300)
xgb_model.fit(X_train, y_train)

print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

print("Training Logistic Regression...")
lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)

print("Training Isolation Forest (Anomaly Detection)...")
iso_model = IsolationForest(contamination=0.05, random_state=42)
iso_model.fit(X_train)

# Evaluate Ensemble (Voting by Average)
preds_xgb = xgb_model.predict_proba(X_test)[:, 1]
preds_rf = rf_model.predict_proba(X_test)[:, 1]
preds_lr = lr_model.predict_proba(X_test)[:, 1]
ensemble_probs = (preds_xgb + preds_rf + preds_lr) / 3.0
ensemble_preds = (ensemble_probs >= 0.5).astype(int)

acc = accuracy_score(y_test, ensemble_preds)
f1 = f1_score(y_test, ensemble_preds)
auc = roc_auc_score(y_test, ensemble_probs)
print(f"Ensemble Accuracy: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")


### 4. Export Metrics and Model

In [ ]:
os.makedirs("models", exist_ok=True)

# Save Metrics
metrics = {
    "model": "Ensemble_B200_Voting",
    "accuracy": float(acc),
    "f1_score": float(f1),
    "auc": float(auc),
    "features": features
}
with open("models/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved metrics.json")

# Save Models
xgb_model.save_model("models/xgb_flood_risk.json")
joblib.dump(rf_model, "models/rf_model.pkl")
joblib.dump(lr_model, "models/lr_model.pkl")
joblib.dump(iso_model, "models/iso_model.pkl")
print("Saved all models to models/ directory.")
